# 02_7. Checks requested at the review of stage 2 / Проверки по замечаниям к этапу 2

**EN.** Answers to the lecturer's review of 2026-10-25, with numbers:
1. Did excluding the 2017-hurricane rows bring the validation event rate closer to test?
2. Why is validation AUC higher than in the stage-1 leakage check (0.89 → 0.93 for Y1 on the same 69 features)? The question refers to the earlier run; the answer below also covers the current one.
3. Investors in the 2007–2009 crisis: does the sign of `occupancy_status` flip?
4. Where was validation used for selection decisions — and the same comparisons **without validation**: cross-validation inside train (GroupKFold by loan).

Current setup: the Y1 population is loans **current at the observation date t** (DPD = 0) that have never been in default before t (only the first default counts), and the 10 delinquency-history features (`dpd`, `n_30p_6m`, `ever_30p` and others) are not candidates, so the model has to warn before any delinquency is visible: 59 candidates, 18 selected. The stage-1 audit used in sections 1–2 has been recomputed on this population and these 59 candidates, so its numbers are comparable with stage 2.

The test set is still not opened; the only test numbers shown are the aggregate event rates already published in the stage-1 audit. The crisis period is a stress scenario: used here only descriptively, nothing is trained on it.

**RU.** Ответы на замечания преподавателя от 2026-10-25, с цифрами:
1. Приблизилась ли доля событий validation к test после исключения наблюдений с ураганами 2017 г.?
2. Почему AUC на validation выше, чем в проверке на утечку этапа 1 (0,89 → 0,93 для Y1 на тех же 69 признаках)? Вопрос относится к предыдущему прогону; ответ ниже охватывает и текущий.
3. Инвесторы в кризис 2007–2009 гг.: меняется ли знак `occupancy_status`?
4. Где validation использовалась для решений об отборе — и те же сравнения **без validation**: кросс-валидация внутри train (GroupKFold по кредиту).

Текущая постановка: популяция Y1 — кредиты **без просрочки на дату наблюдения t** (DPD = 0), ни разу не бывшие в дефолте до t (учитывается только первый дефолт), а 10 признаков истории просрочек (`dpd`, `n_30p_6m`, `ever_30p` и другие) не входят в кандидаты, поэтому модель должна предупреждать до того, как просрочка стала видна: 59 кандидатов, отобрано 18. Аудит этапа 1, на который опираются разделы 1–2, пересчитан на этой популяции и этих 59 кандидатах, поэтому его цифры сравнимы с этапом 2.

Test по-прежнему не открывается; единственные цифры test здесь — агрегированные доли событий, уже опубликованные в аудите этапа 1. Кризисный период — стресс-сценарий: используется только описательно, ничего на нём не обучается.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import gc
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})

import json
from xai_ews.config import feature_config
from xai_ews.data.features import candidate_features
from xai_ews.data.panel import scan_panel
from xai_ews.features.data import TASKS
from xai_ews.features.evaluation import compare_models
from xai_ews.features.importance import cv_oof_logits
from xai_ews.paths import DATA_AUDIT_DIR, FEATURE_SELECTION_DIR

CFG = feature_config()
SEL = CFG["selected"]

### 1. Hurricane exclusion: validation vs test event rates / Исключение ураганов: доли событий validation и test

**EN.** Event rates per observation from the stage-1 audit: validation with and without the flagged rows, the unaffected states at each validation date, and test (aggregate, stage 1). Both tables are on the current Y1 population (loans current at t, never in default before t).

**RU.** Доли событий по наблюдениям из аудита этапа 1: validation с отмеченными наблюдениями и без них, незатронутые штаты на каждой дате validation и test (агрегат этапа 1). Обе таблицы — на текущей популяции Y1 (кредиты без просрочки на t, не бывшие в дефолте до t).

In [2]:
r = lambda n: pd.read_csv(DATA_AUDIT_DIR / n)
wv = r("events_validation_with_without_hurricanes.csv").set_index("validation_set")
test = r("events_by_split.csv").set_index("split_main").loc["test"]
rates = pd.DataFrame({
    "validation, all rows": wv.loc["all", [f"{t}_rate" for t in TASKS]].to_numpy(),
    "validation, without hurricane rows (main)": wv.loc["without_hurricane_rows", [f"{t}_rate" for t in TASKS]].to_numpy(),
    "test (stage-1 aggregate)": test[[f"{t}_rate" for t in TASKS]].to_numpy()}, index=[t.upper() for t in TASKS]).T * 100
display(rates.round(3).rename(columns=lambda c: c + ", %"))
by_date = r("events_validation_hurricanes.csv")
by_date.pivot(index="obs_month", columns="state_group", values=[f"{t}_rate" for t in TASKS]).mul(100).round(3)

,"Y1, %"
"validation, all rows",0.3220
"validation, without hurricane rows (main)",0.2000
test (stage-1 aggregate),0.2110


y1_rate             
state_group TX/FL/PR/VI other states
obs_month                           
2016-06          0.2170       0.1970
2016-09          0.2460       0.1870
2016-12          1.5800       0.1910
2017-03          1.9350       0.2120

**EN. How to read and conclusion.**
- **Yes, closer.** Validation 0.322 % with the hurricane rows (6,472 events on 2,010,979 observations) and 0.200 % without them (3,706 events on 1,854,380 observations), test 0.211 %: the gap shrinks from +0.11 to −0.01 p.p. (train 0.12 %). The small remaining difference is a period shift: in 2018 (test) the event rate is slightly higher than in the unaffected states in 2016–2017. This is why the calibration criterion of H4 is relative. In the earlier population (DPD 0–2, 69 features), to which the review question referred, the exclusion overshot: 0.509 % → 0.379 % against test 0.445 % (gap +0.06 → −0.07 p.p.).
- **The main argument for excluding the hurricane rows is inside validation itself** (second table): in the unaffected states the rate is about the same at every validation date (0.19–0.21 %), while in TX/FL/PR/VI it jumps to 1.58–1.94 %, 8–9 times the other states, *only* at the two dates whose 12-month label windows cover August–September 2017 (before that 0.22–0.25 %). This is a difference-in-differences pattern of an external shock, not a property of the borrowers. Together the 156,599 flagged rows carry 2,766 events (1.77 %), about 9 times the rate of the other validation rows (0.200 %).
- The rows are not deleted: every validation table in stage 2 also reports the metrics *with* them (column `auc_with_hurricanes`), as a robustness check.
- **Formal rule for stage 3:** areas from the FEMA disaster declarations × label windows that cover the incident period, applied the same way to validation, test and the stress samples, decided before looking at test outcomes. The state-level flag used here (TX, FL, PR, VI) is the version of this rule available from the data at state level.

**RU. Как читать и вывод.**
- **Да, ближе.** Validation 0,322 % с наблюдениями урагана (6 472 события на 2 010 979 наблюдений) и 0,200 % без них (3 706 событий на 1 854 380 наблюдений), test 0,211 %: разрыв сокращается с +0,11 до −0,01 п.п. (train 0,12 %). Небольшая оставшаяся разница — сдвиг периода: в 2018 г. (test) доля событий немного выше, чем в незатронутых штатах в 2016–2017 гг. Поэтому критерий калибровки в H4 относительный. В прежней популяции (DPD 0–2, 69 признаков), к которой относился вопрос рецензии, исключение «перелетало» test: 0,509 % → 0,379 % при test 0,445 % (разрыв +0,06 → −0,07 п.п.).
- **Главный аргумент за исключение наблюдений урагана — внутри самой validation** (вторая таблица): в незатронутых штатах доля событий почти одинакова на всех датах validation (0,19–0,21 %), а в TX/FL/PR/VI она вырастает до 1,58–1,94 %, в 8–9 раз выше остальных штатов, *только* на двух датах, чьи 12-месячные окна меток захватывают август–сентябрь 2017 г. (до этого 0,22–0,25 %). Это картина «разность разностей» от внешнего шока, а не свойство заёмщиков. Всего на 156 599 отмеченных наблюдений приходится 2 766 событий (1,77 %), примерно в 9 раз больше, чем в остальной validation (0,200 %).
- Наблюдения не удаляются бесследно: все таблицы validation этапа 2 приводят метрики и *с* ними (колонка `auc_with_hurricanes`) как проверку устойчивости.
- **Формальное правило на этап 3:** территории из деклараций бедствий FEMA × окна меток, захватывающие период бедствия, — применяются одинаково к validation, test и стресс-выборкам и фиксируются до просмотра исходов test. Используемый здесь флаг на уровне штатов (TX, FL, PR, VI) — версия этого правила, доступная по данным на уровне штата.

### 2. Why validation AUC is higher than in stage 1 / Почему AUC на validation выше, чем на этапе 1

**EN.** The same kind of model (CatBoost 300 × 6, full train r = 0.2) in four steps: the stage-1 leakage check, the stage-2 model on all 59 candidates on validation with the hurricane rows, the same model without the hurricane rows, and the selected 18. All four rows use the current population (loans current at t, never in default before t) and the same 59 candidates.

**RU.** Одна и та же модель (CatBoost 300 × 6, полный train r = 0,2) в четырёх шагах: проверка на утечку этапа 1, модель этапа 2 на всех 59 кандидатах на validation с наблюдениями урагана, та же модель без наблюдений урагана и отобранные 18. Все четыре строки — на текущей популяции (кредиты без просрочки на t, не бывшие в дефолте до t) и тех же 59 кандидатах.

In [3]:
lc = json.loads((DATA_AUDIT_DIR / "leakage_check.json").read_text())
abl = pd.read_csv(FEATURE_SELECTION_DIR / "ablation.csv").set_index(["task", "model"])
steps = pd.DataFrame({t: [lc[t]["validation_auc"], abl.loc[(t, "all"), "auc_with_hurricanes"],
                          abl.loc[(t, "all"), "auc"], abl.loc[(t, "selected"), "auc"]] for t in TASKS},
                     index=["stage 1 leakage check: 59 candidates, validation with hurricane rows",
                            "stage 2: 59 candidates, validation with hurricane rows",
                            "stage 2: 59 candidates, validation without hurricane rows",
                            f"stage 2: selected {len(SEL['features'])}, validation without hurricane rows"])
steps.round(4)

,y1
"stage 1 leakage check: 59 candidates, validation with hurricane rows",0.8195
"stage 2: 59 candidates, validation with hurricane rows",0.8220
"stage 2: 59 candidates, validation without hurricane rows",0.8509
"stage 2: selected 18, validation without hurricane rows",0.8538


**EN. Conclusion.** All four rows are comparable. The stage-1 leakage check (0.820) and the stage-2 model on the same 59 candidates and the same validation with the hurricane rows (0.822) agree; the rise to 0.851 comes from removing the hurricane rows. The earlier run behind the review question (0.89 → 0.93; DPD 0–2, 69 features including the delinquency history) showed the same pattern: 0.891 in stage 1, 0.893 in stage 2 with the hurricane rows, 0.930 without them. All current numbers are lower than in that run because the task is harder: only loans current at t that have never been in default, and no delinquency history among the features — the model has to warn before any delinquency is visible.
Step by step:
- **Removing the hurricane rows adds +0.029** (0.822 → 0.851 for all 59 candidates). The hurricane-driven defaults are unpredictable from borrower data (an external shock), so they lower AUC.
- **Feature selection costs nothing:** the selected 18 give 0.8538 vs 0.8509 for all 59 (the selected set is 0.003 better, 02_6).

**RU. Вывод.** Все четыре строки сравнимы. Проверка на утечку этапа 1 (0,820) и модель этапа 2 на тех же 59 кандидатах и той же validation с наблюдениями урагана (0,822) совпадают; рост до 0,851 даёт исключение наблюдений урагана. Предыдущий прогон, к которому относился вопрос рецензии (0,89 → 0,93; DPD 0–2, 69 признаков вместе с историей просрочек), показывал ту же картину: 0,891 на этапе 1, 0,893 на этапе 2 с наблюдениями урагана, 0,930 без них. Все текущие цифры ниже, чем в том прогоне, потому что задача сложнее: только кредиты без просрочки на t, ни разу не бывшие в дефолте, и никакой истории просрочек среди признаков — модель должна предупреждать до того, как просрочка стала видна.
По шагам:
- **Исключение наблюдений урагана добавляет +0,029** (0,822 → 0,851 для всех 59 кандидатов). Дефолты, вызванные ураганом, нельзя предсказать по данным заёмщика (внешний шок), поэтому они снижают AUC.
- **Отбор признаков ничего не стоит:** отобранные 18 дают 0,8538 против 0,8509 для всех 59 (отобранный набор лучше на 0,003, 02_6).

### 3. Investors in the 2007–2009 crisis / Инвесторы в кризис 2007–2009 гг.

**EN.** GFC stress scenario (observation dates 2007-03…2010-12, vintages 2006–2008, 15 % sample, unweighted; Y1 population: loans current at t, never in default before t): event rates of investor (I) vs principal-residence (P) loans, overall and within FICO and original-LTV bands. In train (2013–2015) investors were *safer* (notebook 02_3). If the sign flips here, the effect depends on the regime.

**RU.** Стресс-сценарий кризиса (даты наблюдения 2007-03…2010-12, поколения 2006–2008, 15 %-я выборка, без весов; популяция Y1 — кредиты без просрочки на t, не бывшие в дефолте до t): доли событий у инвестиционных (I) и основного жилья (P), в целом и внутри полос FICO и LTV при выдаче. В train (2013–2015) инвесторы были *безопаснее* (ноутбук 02_3). Если здесь знак перевернётся, эффект зависит от режима.

In [4]:
gfc = (scan_panel().filter(pl.col("stress") == "gfc")
       .select("loan_identifier", "occupancy_status", "borrower_credit_score_at_origination",
               "original_loan_to_value_ratio_ltv", *[c for t, pop in TASKS.items() for c in (pop, t)]).collect())
gfc = gfc.with_columns(
    pl.lit("all").alias("all"),
    (pl.col("borrower_credit_score_at_origination") // 40 * 40).clip(600, 800).cast(pl.Int32).alias("fico_band"),
    pl.col("original_loan_to_value_ratio_ltv").cut([60, 70, 80, 90]).cast(pl.String).alias("ltv_band"))
print("GFC rows:", gfc.height, "| loans:", gfc["loan_identifier"].n_unique())
gfc_tabs = []
for task, pop in TASKS.items():
    d = gfc.filter(pl.col(pop) & pl.col(task).is_not_null() & pl.col("occupancy_status").is_in(["I", "P"]))
    for by in ["all", "fico_band", "ltv_band"]:
        g = (d.group_by("occupancy_status", by).agg(pl.len().alias("n"), pl.col(task).sum().alias("events"),
                                                    pl.col(task).mean().alias("rate"))
             .to_pandas().pivot(index=by, columns="occupancy_status", values=["rate", "events"]))
        g.columns = [f"{a}_{b}" for a, b in g.columns]
        g["I_minus_P_pp"] = 100 * (g["rate_I"] - g["rate_P"])
        gfc_tabs.append(g.reset_index().rename(columns={by: "band"}).assign(task=task, by=by))
gfc_tab = pd.concat(gfc_tabs, ignore_index=True)
gfc_tab.to_csv(FEATURE_SELECTION_DIR / "occupancy_gfc.csv", index=False)
gfc_tab.set_index(["task", "by", "band"]).round(4)

GFC rows: 1606503 | loans: 176163


rate_I  rate_P   events_I    events_P  I_minus_P_pp
task by        band                                                            
y1   all        all         0.0266  0.0264 3,886.0000 34,674.0000        0.0250
     fico_band NaN          0.0165  0.0280     7.0000    123.0000       -1.1541
               600.0000     0.0635  0.0582   279.0000  6,002.0000        0.5266
               640.0000     0.0529  0.0476   671.0000  8,698.0000        0.5235
               680.0000     0.0377  0.0345 1,017.0000  9,084.0000        0.3244
               720.0000     0.0283  0.0225 1,089.0000  6,362.0000        0.5759
               760.0000     0.0144  0.0105   716.0000  3,827.0000        0.3931
               800.0000     0.0080  0.0051   107.0000    578.0000        0.2948
     ltv_band   (-inf, 60]  0.0096  0.0113   212.0000  3,740.0000       -0.1709
                (60, 70]    0.0293  0.0278   706.0000  5,642.0000        0.1569
                (70, 80]    0.0274  0.0284 2,091.0000 15,670.0000       -0.1046
                (80, 90]    0.0373  0.0417   875.0000  5,730.0000       -0.4356
                (90, inf]   0.0455  0.0419     2.0000  3,892.0000        0.3533

**EN. How to read and conclusion.** 1.6 M observations of the 2006–2008 vintages in 2007–2010 (before the population filter; 1.54 M of them are in the Y1 population); in the Y1 population the event rate is 2.60 %, about 20 times higher than in train (0.12 %).
- **Within every FICO band the sign flips:** in the crisis investors are *riskier* than principal residences — +0.29…+0.58 p.p. (e.g. FICO 720–759: 2.8 % vs 2.3 %). The row with missing FICO (7 investor events) is too small to read. In 2013–2015 investors had a *lower* event rate: 0.07 % vs 0.13 % on train (0.10 % vs 0.21 % on validation), and also within the LTV bands up to 80 (notebook 02_3).
- Overall the two groups are almost equal in the crisis (2.66 % vs 2.64 %), and by LTV band the sign is mixed (−0.44…+0.16 p.p.; the band above 90, +0.35, has only 2 investor events): investors of those vintages had higher credit scores (69 % of investor observations have FICO ≥ 720, against 58 % of principal residences), so the raw comparison mixes in composition. Holding the credit score fixed — which is what the model does — the investor effect reverses.
- **Conclusion:** the effect of `occupancy_status` (one of the selected 18) **depends on the regime**. In calm years with strict post-crisis underwriting, investors who got a loan are a selected strong group; in a downturn they default more (strategic default on a property they do not live in). The feature stays, without a sign constraint, and is flagged for **monitoring**: its contribution should be checked in the stress tests (stage 5) and in model monitoring, because in a downturn its direction changes.

**RU. Как читать и вывод.** 1,6 млн наблюдений поколений 2006–2008 гг. в 2007–2010 гг. (до фильтра популяции; из них 1,54 млн — в популяции Y1); в популяции Y1 доля событий 2,60 %, примерно в 20 раз выше, чем в train (0,12 %).
- **Внутри каждой полосы FICO знак меняется:** в кризис инвесторы *рискованнее* основного жилья — +0,29…+0,58 п.п. (например, FICO 720–759: 2,8 % против 2,3 %). Строка с пропущенным FICO (7 событий у инвесторов) слишком мала, чтобы её читать. В 2013–2015 гг. у инвесторов доля событий была *ниже*: 0,07 % против 0,13 % на train (0,10 % против 0,21 % на validation), и так же внутри полос LTV до 80 (ноутбук 02_3).
- В целом в кризис группы почти равны (2,66 % против 2,64 %), а по полосам LTV знак смешанный (−0,44…+0,16 п.п.; в полосе выше 90, +0,35, у инвесторов всего 2 события): у инвесторов этих поколений были выше рейтинги (FICO ≥ 720 у 69 % наблюдений инвесторов против 58 % у основного жилья), и простое сравнение смешивает в себе состав. При фиксированном рейтинге — а модель сравнивает именно так — эффект инвестора разворачивается.
- **Вывод:** эффект `occupancy_status` (входит в отобранные 18) **зависит от режима**. В спокойные годы при строгом посткризисном андеррайтинге инвесторы, получившие кредит, — отобранная сильная группа; в спад они дефолтят чаще (стратегический дефолт по объекту, в котором не живут). Признак остаётся без ограничения знака и помечается для **мониторинга**: его вклад нужно проверить на стресс-тестах (этап 5) и при мониторинге модели, потому что в спад его направление меняется.

### 4. Where validation was used / Где использовалась validation

| Step | Validation used for the decision? | How |
| --- | --- | --- |
| 2 IV | no | decision on train IV; validation IV only reported |
| 6 stability | **yes** | PSI train → validation; macro: years 2016–2017 are validation |
| 7 sign | partly | direction on train decides; validation direction supports the review |
| 4–5 redundancy | no | correlations and VIF on train |
| 8 importance | **yes** | permutation importance on validation (Boruta on train) |
| 9 ablation | **yes** | group rule on validation ΔAUC (geography group removed) |
| non-inferiority | **yes** | 18 vs 59 on validation |

**EN.** Stability *must* look at a later period — that is what it checks. But steps 8–9 and the comparison of 18 vs 59 used the same validation on which they are reported, so those validation numbers are optimistic. Below the model-based comparisons are repeated **inside train only**: 3-fold GroupKFold by loan (all snapshots of a loan in one fold), out-of-fold predictions, cluster bootstrap by loan. This checks the same period (2013–2015), not out of time, but it does not touch validation at all.

**RU.** Стабильность *должна* смотреть на более поздний период — именно это она и проверяет. Но шаги 8–9 и сравнение 18 и 59 признаков использовали ту же validation, на которой о них сообщается, поэтому эти цифры на validation оптимистичны. Ниже модельные сравнения повторены **только внутри train**: 3-кратная GroupKFold по кредиту (все снимки кредита — в одном фолде), out-of-fold прогнозы, кластерный bootstrap по кредиту. Это проверка на том же периоде (2013–2015), а не на другом, но validation она не затрагивает вовсе.

In [5]:
GEO = ["property_state", "metropolitan_statistical_area_msa_or_metropolitan_statistical_division_area_msda", "zip_code_short"]
sets = {"selected": SEL["features"], "all": candidate_features(),
        **({"with_geography": SEL["features"] + GEO} if not set(GEO) <= set(SEL["features"]) else {})}
cv = {}
for task in TASKS:
    oof = cv_oof_logits(task, sets, CFG["rules"]["importance"]["catboost"], n_folds=3, seed=CFG["data"]["seed"])
    res = compare_models(oof, task, "selected", b=CFG["rules"]["ablation"]["bootstrap_b"], seed=CFG["data"]["seed"])
    print(task, "train counts:", res.attrs["counts"])
    cv[task] = res.assign(task=task)
    gc.collect()
cv = pd.concat(cv.values(), ignore_index=True)
cv.to_csv(FEATURE_SELECTION_DIR / "cv_within_train.csv", index=False)
cv[["task", "model", "auc", "auc_lo", "auc_hi", "delta_auc", "delta_lo", "delta_hi", "delong_p"]]

y1 train counts: {'rows': 2294292, 'loans': 1239248, 'events_obs': 14062, 'event_loans': 7068, 'one_snapshot_rows': 1239248, 'one_snapshot_events': 6223}


,task,model,auc,auc_lo,auc_hi,delta_auc,delta_lo,delta_hi,delong_p
0,y1,selected,0.8578,0.8535,0.8624,NaN,NaN,NaN,NaN
1,y1,all,0.8561,0.8517,0.8606,-0.0017,-0.0031,-0.0004,0.0106
2,y1,with_geography,0.8552,0.8509,0.8600,-0.0026,-0.0040,-0.0013,0.0001


**EN. Conclusion.** Without using validation at all, the model-based conclusion of stage 2 holds. Out-of-fold AUC on train (3-fold GroupKFold by loan; 2.29 M observations of the r = 0.2 train, 1.24 M loans, 14,062 events on 7,068 loans; cluster bootstrap by loan, DeLong on one snapshot per loan):
- **The selected 18 are non-inferior to all 59**: AUC 0.858 vs 0.856, ΔAUC(all − selected) = −0.0017 [−0.0031; −0.0004], DeLong p = 0.011. The upper bound of the interval is below the non-inferiority margin of 0.005, so the criterion holds inside train as well; as on validation (−0.0029 [−0.0051; −0.0010], DeLong p = 0.003), the selected set is even slightly better than all 59.
- **Geography** (removed at step 9): the selected set plus the whole geography group (state, MSA and ZIP3) gives 0.855 vs 0.858, ΔAUC = −0.0026 [−0.0040; −0.0013], DeLong p < 0.001 — significantly worse inside train as well. So geography makes the model worse both within the train period and on the later validation period (02_6, −0.0036): it is not only a change between periods. A likely reason: state, MSA and especially ZIP3 have many categories with few events, and their target statistics in CatBoost partly learn noise.
The clean final assessment remains test (stage 3).

**RU. Вывод.** Без использования validation модельный вывод этапа 2 сохраняется. Out-of-fold AUC на train (3-кратная GroupKFold по кредиту; 2,29 млн наблюдений train с r = 0,2, 1,24 млн кредитов, 14 062 события на 7 068 кредитах; кластерный bootstrap по кредиту, DeLong на одном снимке на кредит):
- **Отобранные 18 не хуже всех 59**: AUC 0,858 против 0,856, ΔAUC(все − отобранные) = −0,0017 [−0,0031; −0,0004], DeLong p = 0,011. Верхняя граница интервала ниже порога «не хуже» 0,005, поэтому критерий выполняется и внутри train; как и на validation (−0,0029 [−0,0051; −0,0010], DeLong p = 0,003), отобранный набор даже немного лучше всех 59.
- **География** (удалена на шаге 9): отобранный набор плюс вся группа «география» (штат, MSA и ZIP3) даёт 0,855 против 0,858, ΔAUC = −0,0026 [−0,0040; −0,0013], DeLong p < 0,001 — и внутри train значимо хуже. Значит, география ухудшает модель и внутри периода train, и на более позднем периоде validation (02_6, −0,0036): дело не только в смене периода. Вероятная причина: у штата, MSA и особенно ZIP3 много категорий с небольшим числом событий, и их целевые статистики в CatBoost отчасти запоминают шум.
Чистая итоговая оценка — по-прежнему test (этап 3).